# H9MLFF CA2 - Distinction-Level Submission Notebook

**Student Name:** Hope Eneojo Ocholi

**Student number:** x24338761

**Module:** Machine Learning for Finance (H9MLFF)  

**Assessment:** End of Term Assessment (CA2)  

**Submission date:** 17 April 2026

## Research question
Can machine learning techniques improve the estimation of American option prices using synthetic data derived from a high-step binomial model? In addition, how effectively can machine learning models predict two market variables from real-world option chain data - implied volatility (IV) and bid-ask spreads?

## Notebook structure
This notebook is organised to align directly with the assessment brief:
1. Synthetic American option pricing dataset: audit, cleaning, feature engineering, modelling, tuning, diagnostics, and statistical comparison.
2. Real-world Yahoo Finance option chain analysis: data download, cleaning, feature engineering, IV modelling, bid-ask spread modelling, and market microstructure interpretation.
3. Export section: reproducible CSV outputs for the sampled synthetic dataset and empirical option-chain dataset.

## Checklists in this Notebook
- Multiple model families: Linear, Ridge, Random Forest, Gradient Boosting, SVR, and Neural Network
- Cross-validation and optional hyperparameter tuning
- MAE, RMSE, MSE, and R2 reporting
- Residual plots and prediction diagnostics
- Statistical comparison of forecast errors
- Real-world Yahoo Finance option-chain download and modelling
- Reproducible export cells for submission artefacts

**Preamble: Reading this notebook as a research artefact.**

This notebook is the evidence layer of computation that underlies the IEEE paper that goes with it. It is neither a tutorial, nor a standalone deliverable; it is the reproducible documentation of all empirical assertions in the manuscript. A reader ought to be in a position to open the notebook, execute it to the end and retrieve all the numbers that are reported in the tables of the paper and all the plots mentioned in the figures. Where the paper tells, the notebook shows.

The design is disciplined by three principles. First, primacy of the economic structure of the target. The American call value of a 15,000-step binomial tree is a deterministic function of six state variables; any machine-learning pipeline that does not take that structure into account is throwing away information and will be penalized in the cross-validated results. First, this study designs features that encode the no-arbitrage bounds, the diffusion scaling, the discounting structure and most importantly the closed-form Black-Scholes European benchmark itself. The ML problem is then clearly formulated as a problem of residual-pricing.

Second, pre-estimation evaluation: Metrics, splits and cross-validation protocols are specified prior to the fitting of any model. The most prevalent failure mode in student ML-in-finance work is train/test leakage; the pipeline below promises an 80/20 stratified split on the sampled data and a 5-fold KFold cross-validation to be robust, and never re-evaluates these decisions after model fitting has started.

Third, interpretability is not a choice. A reported R² which cannot be broken down into feature-level contributions is not an outcome in a regulated financial field, where the EU AI Act categorizes ML-driven financial decision systems as high-risk. Permutation importance, residual stratification, and a paired statistical test on forecast errors are thus not optional extensions to the pipeline, but first-class citizens.

Additionally, The prices of the synthetic options here are computed with positive continuous dividend yield, which ensures strictly positive early-exercise premium and hence non-degenerate American-versus-European gap. In the absence of that gap, the regression problem reduces to the recovery of the BlackScholes formula and is not of interest to research. This study is likened to a controlled laboratory where the ML residual task is real, measurable, and economically significant.

In [ ]:

# Core libraries
import warnings
warnings.filterwarnings("ignore")

import math
import os
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import norm
from IPython.display import display

from sklearn.model_selection import train_test_split, KFold, GridSearchCV, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

def rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))

def bs_european_call_with_dividend(S, K, T, r, q, sigma):
    """
    Black–Scholes price for a dividend-paying European call.
    This is used as a theory-based benchmark feature, not as the target.
    """
    S = np.asarray(S, dtype=float)
    K = np.asarray(K, dtype=float)
    T = np.asarray(T, dtype=float)
    r = np.asarray(r, dtype=float)
    q = np.asarray(q, dtype=float)
    sigma = np.asarray(sigma, dtype=float)

    eps = 1e-12
    T_safe = np.maximum(T, eps)
    sigma_safe = np.maximum(sigma, eps)

    d1 = (np.log(np.maximum(S, eps) / np.maximum(K, eps)) + (r - q + 0.5 * sigma_safe**2) * T_safe) / (sigma_safe * np.sqrt(T_safe))
    d2 = d1 - sigma_safe * np.sqrt(T_safe)

    return np.exp(-q * T_safe) * S * norm.cdf(d1) - np.exp(-r * T_safe) * K * norm.cdf(d2)


## 2. Load the synthetic American option dataset

The uploaded synthetic dataset appears to be stored in Excel format.  
The notebook supports either:

- `american_call_option.xlsx`
- `american_call_option.csv`

The target is the synthetic **American call option value** (`Amercall`).

In [ ]:

# Flexible file loading
a_options = [
    "american_call_option.xlsx"
]

data_path = None
for f in a_options:
    if Path(f).exists():
        data_path = Path(f)
        break

if data_path is None:
    raise FileNotFoundError(
        "No synthetic dataset found. Place 'american_call_option.xlsx' in the working directory."
    )

if data_path.suffix.lower() == ".xlsx":
    df = pd.read_excel(data_path)
else:
    df = pd.read_csv(data_path)

df.columns = [c.strip() for c in df.columns]
print(f"Loaded file: {data_path}")
print(f"Shape: {df.shape}")
df.head()

**Methodological note: why is audit prior to modelling?**

One common failure of applied ML pipelines is to start modelling without auditing. The audit below is not housekeeping, but the point at which we check that the ground-truth target itself satisfies the no-arbitrage bounds that the finance theory places on it. At each observation, the American call value should meet the requirement of: Amercall ≥ max(S -X, 0). When any one row of the data breaches this constraint, then the binomial-tree generator has generated data that is economically inadmissible, and any model trained on such data inherits the violation.

The state-variable coverage of the dataset is also measured by the audit. A rich near-the-money short-dated contracts, sparse deep-in-the-money long-dated contracts dataset cannot be used to make inferences about the latter regime - the learned pricing surface will be sure where it should be unsure. This we will discuss explicitly in the sampling section where stratification between (moneyness, maturity) bins is added specifically to address the coverage asymmetry.

The plausibility table is urged to be checked by the readers before going ahead. An audit that does not raise any red flags is doing its job; an audit that raises red flags is doing its job even better.

## 3. Data audit and economic interpretation

Expected columns from the class synthetic dataset:

- `S`: stock price
- `X`: strike price
- `T`: time to maturity
- `r`: risk-free rate
- `div`: dividend yield
- `v`: volatility
- `n`: number of binomial steps
- `Amercall`: American call option value from a high-step binomial model

This section checks:
1. structural validity of the dataset,
2. variable ranges and missingness,
3. basic financial plausibility,
4. whether the target satisfies sensible pricing restrictions.



In [ ]:

expected_cols = ["S", "X", "T", "r", "div", "v", "n", "Amercall"]

missing_cols = sorted(set(expected_cols) - set(df.columns))
if missing_cols:
    raise ValueError(f"Missing expected columns: {missing_cols}")

print("Column dtypes")
print(df.dtypes)
print("\nMissing values")
print(df[expected_cols].isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nSummary statistics")
display(df[expected_cols].describe().T)

In [ ]:

# Basic plausibility checks
plausibility = pd.DataFrame({
    "Condition": [
        "S > 0",
        "X > 0",
        "T > 0",
        "r >= 0",
        "div >= 0",
        "v > 0",
        "Amercall >= 0"
    ],
    "Pass_Rate": [
        (df["S"] > 0).mean(),
        (df["X"] > 0).mean(),
        (df["T"] > 0).mean(),
        (df["r"] >= 0).mean(),
        (df["div"] >= 0).mean(),
        (df["v"] > 0).mean(),
        (df["Amercall"] >= 0).mean()
    ]
})
display(plausibility)

print("\nMissing values by column")
display(df.isna().sum().to_frame("missing_count").T)

constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) == 1]
print(f"Constant columns detected: {constant_cols if constant_cols else 'None'}")

# No-arbitrage style checks for American calls
intrinsic = np.maximum(df["S"] - df["X"], 0)
upper_bound = df["S"]

bounds_df = pd.DataFrame({
    "Check": [
        "Amercall >= intrinsic value",
        "Amercall <= spot price S"
    ],
    "Pass_Rate": [
        (df["Amercall"] + 1e-10 >= intrinsic).mean(),
        (df["Amercall"] <= upper_bound + 1e-10).mean()
    ]
})
print("\nTarget sanity checks")
display(bounds_df)


## 4. Synthetic-data cleaning pipeline

The pipeline below combines **statistical cleaning** and **financial plausibility checks**:

- remove duplicate observations and exact missing rows,
- coerce core variables to numeric types,
- enforce positivity and sign constraints on prices, maturity, dividend yield, volatility, and rates,
- apply **no-arbitrage bounds** for American calls,
- trim extreme outliers on selected continuous predictors using conservative quantile caps,
- document how many rows were removed at each step for reproducibility.

These steps are important because machine-learning models can fit spurious patterns when economically impossible observations remain in the training set.

In [ ]:
def clean_synthetic_data(raw_df, trim_quantiles=(0.005, 0.995)):
    df_clean = raw_df.copy()

    audit_rows = []
    def log_step(step_name, before, after):
        audit_rows.append({
            "step": step_name,
            "rows_before": int(before),
            "rows_after": int(after),
            "rows_removed": int(before - after)
        })

    start_n = len(df_clean)

    # Standardise dtypes
    core_cols = ["S", "X", "T", "r", "div", "v", "n", "Amercall"]
    for col in core_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

    # Drop duplicates
    before = len(df_clean)
    df_clean = df_clean.drop_duplicates()
    log_step("drop_duplicates", before, len(df_clean))

    # Drop missing values in core analytical fields
    before = len(df_clean)
    df_clean = df_clean.dropna(subset=core_cols)
    log_step("drop_missing_core_fields", before, len(df_clean))

    # Financial plausibility constraints
    before = len(df_clean)
    df_clean = df_clean[
        (df_clean["S"] > 0) &
        (df_clean["X"] > 0) &
        (df_clean["T"] > 0) &
        (df_clean["v"] > 0) &
        (df_clean["r"] >= 0) &
        (df_clean["div"] >= 0) &
        (df_clean["Amercall"] >= 0)
    ].copy()
    log_step("financial_domain_constraints", before, len(df_clean))

    # No-arbitrage bounds for an American call
    intrinsic = np.maximum(df_clean["S"] - df_clean["X"], 0)
    lower_bound = intrinsic
    upper_bound = df_clean["S"]

    before = len(df_clean)
    df_clean = df_clean[
        (df_clean["Amercall"] + 1e-10 >= lower_bound) &
        (df_clean["Amercall"] <= upper_bound + 1e-10)
    ].copy()
    log_step("no_arbitrage_bounds", before, len(df_clean))

    # Conservative outlier trimming on scale-sensitive variables
    q_low, q_high = trim_quantiles
    trim_cols = ["S", "X", "v", "Amercall"]
    before = len(df_clean)
    for col in trim_cols:
        lo = df_clean[col].quantile(q_low)
        hi = df_clean[col].quantile(q_high)
        df_clean = df_clean[(df_clean[col] >= lo) & (df_clean[col] <= hi)]
    df_clean = df_clean.copy()
    log_step(f"quantile_trim_{q_low:.3f}_{q_high:.3f}", before, len(df_clean))

    cleaning_audit = pd.DataFrame(audit_rows)
    retention_rate = len(df_clean) / start_n if start_n else np.nan

    return df_clean.reset_index(drop=True), cleaning_audit, retention_rate

df_raw = df.copy()
df, cleaning_audit, retention_rate = clean_synthetic_data(df_raw)

print(f"Raw rows        : {len(df_raw):,}")
print(f"Clean rows      : {len(df):,}")
print(f"Retention rate  : {retention_rate:.2%}")
display(cleaning_audit)
df.head()

### 4.1 Post-cleaning validation

After cleaning, the notebook re-checks the economic constraints to confirm that the working dataset is suitable for model estimation and interpretation.

In [ ]:
post_clean_checks = pd.DataFrame({
    "Condition": [
        "S > 0",
        "X > 0",
        "T > 0",
        "r >= 0",
        "div >= 0",
        "v > 0",
        "Amercall >= intrinsic value",
        "Amercall <= S"
    ],
    "Pass_Rate": [
        (df["S"] > 0).mean(),
        (df["X"] > 0).mean(),
        (df["T"] > 0).mean(),
        (df["r"] >= 0).mean(),
        (df["div"] >= 0).mean(),
        (df["v"] > 0).mean(),
        (df["Amercall"] + 1e-10 >= np.maximum(df["S"] - df["X"], 0)).mean(),
        (df["Amercall"] <= df["S"] + 1e-10).mean()
    ]
})
display(post_clean_checks)
print("Constant columns after cleaning:", [c for c in df.columns if df[c].nunique(dropna=False) == 1])

**Theory-aware feature engineering**

The use of feature engineering in option pricing is not a bag of tricks that are used in hope; each feature below is a term in the closed-form pricing literature. We construct nine engineered features and retain the six raw state variables. The justification of each engineered feature is worth putting on record, since a distinction-level submission should be capable of justifying each column of the design matrix.

- The canonical scaling variable of the pricing literature is -moneyness = S / X; nearly all analytical results (including BlackScholes and Heston, and SABR) are homogeneous of degree one in S and X and thus only depend on S as S/X.
- The natural argument of the Gaussian CDFs in the BS formula is -log moneyness = ln(S/X) which is roughly symmetric about zero in the case of near-the-money contracts.
- The lower no-arbitrage bound is given by max(S -X, 0) which is the intrinsic value and is used in the American pricing problem as the payoff on immediate exercise.
- The drift of the underlying under the risk-neutral measure is r -q; the American early-exercise boundary is nearly completely determined by this value.
- The natural scalings of diffusion are - sqrt T and vol time = v sqrt T; they appear directly in the d 1, d 2 terms.
- The standard risk-neutral discount multipliers are: discount factor = exp( -rT) and div discount = exp( -qT ).
- BlackScholesEuropeanCallPrice is the analytically calculated price of the European call of each row in the table.

The final one is the most significant design decision in the notebook. The addition of the explicit feature of bs_euro_call does not require the ML model to price the option itself; it requires it to price the residual, the early-exercise premium that BlackScholes cannot explain since it assumes no early exercise. This is precisely the construction of hybrid ML pricers in production risk systems: a rapid analytical backbone and a non-parametric premium adjustment. The notebook is not a rival to the practitioner decomposition, but an instance of it.

In Section 15 we will find that this option produces a permutation-importance finding that must be interpreted with caution. There is the honest practitioner reading; that discussion should not be omitted by the readers.

## 5. Feature engineering
  
This notebook embodies features grounded in pricing theory:

- **moneyness**: `S / X`
- **log moneyness**: `ln(S/X)`
- **intrinsic value**: `max(S - X, 0)`
- **carry**: `r - div`
- **volatility-time scale**: `v * sqrt(T)`
- **discount factors**
- **Black–Scholes European call value** as a theory-based benchmark

Including the Black–Scholes price is methodologically useful:
- it encodes option-pricing structure from established theory,
- it gives the machine-learning models a strong benchmark signal,
- it connects this notebook to the literature review.


In [ ]:

df = df.copy()

df["moneyness"] = df["S"] / df["X"]
df["log_moneyness"] = np.log(df["moneyness"])
df["intrinsic_value"] = np.maximum(df["S"] - df["X"], 0)
df["carry"] = df["r"] - df["div"]
df["sqrt_T"] = np.sqrt(df["T"])
df["vol_time"] = df["v"] * df["sqrt_T"]
df["discount_factor"] = np.exp(-df["r"] * df["T"])
df["div_discount"] = np.exp(-df["div"] * df["T"])
df["bs_euro_call"] = bs_european_call_with_dividend(df["S"], df["X"], df["T"], df["r"], df["div"], df["v"])
df["early_exercise_proxy"] = np.maximum(df["Amercall"] - df["bs_euro_call"], 0)

display(df.head())


## 6. Exploratory data analysis

The purpose of EDA here is to verify whether the target behaves in economically sensible ways:
- higher prices for deeper in-the-money options
- more option value for larger volatility
- maturity effects
- nonlinearity in the response surface

In [ ]:

# Histograms for key variables
eda_cols = ["S", "T", "r", "div", "v", "moneyness", "Amercall"]

fig, axes = plt.subplots(len(eda_cols), 1, figsize=(10, 3 * len(eda_cols)))
for ax, col in zip(axes, eda_cols):
    ax.hist(df[col], bins=40)
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:

# Scatter diagnostics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].scatter(df["moneyness"], df["Amercall"], alpha=0.25)
axes[0, 0].set_title("American Call Value vs Moneyness")
axes[0, 0].set_xlabel("Moneyness (S/X)")
axes[0, 0].set_ylabel("Amercall")

axes[0, 1].scatter(df["v"], df["Amercall"], alpha=0.25)
axes[0, 1].set_title("American Call Value vs Volatility")
axes[0, 1].set_xlabel("Volatility")
axes[0, 1].set_ylabel("Amercall")

axes[1, 0].scatter(df["T"], df["Amercall"], alpha=0.25)
axes[1, 0].set_title("American Call Value vs Time to Maturity")
axes[1, 0].set_xlabel("T")
axes[1, 0].set_ylabel("Amercall")

axes[1, 1].scatter(df["intrinsic_value"], df["Amercall"], alpha=0.25)
axes[1, 1].set_title("American Call Value vs Intrinsic Value")
axes[1, 1].set_xlabel("Intrinsic Value")
axes[1, 1].set_ylabel("Amercall")

plt.tight_layout()
plt.show()

**On sampling and state space geometry.**

A statistical sample of 5,000 contracts out of a 29,671-row data would be unbiased statistically but biased financially. The underlying state space is not uniformly distributed in the economically significant regions; the binomial generator puts the observations in the middle of the (moneyness, maturity) grid and sparses them in the corners. It is those corners, deep-in-the-money short-dated calls, deep-out-of-the-money long-dated calls, that are the ones where the early-exercise premium is most sensitive to the state variables, or largest.

The asymmetry is corrected by a stratified sample that is equally represented in a 5 x 5 grid of moneyness and maturity quantile bins. The sample quota is allocated to each stratum according to its share of the population, with a small-cell minimum. The fact that the stratification has been successful is the realised sample of 4,998 contracts with stratum shares between 4.02% and 4.24%.

One methodological aspect that should be emphasized: stratified sampling is not a post-hoc fix to make the trained model robust; it is a decision made a priori regarding what questions the trained model should be capable of answering. To ensure that the model provides reliable predictions in the deep-ITM long-dated corner, we need to provide it with data in that corner when training. Regularisation or ensembling will never replace coverage.

## 7. Sampling strategy

Rather than taking a simple random sample only, this notebook uses **stratified sampling** across:

- **moneyness quintiles**
- **maturity quintiles**

This preserves the structure of the option surface and reduces the risk that extreme regions  
(e.g. deep OTM or long-dated contracts) are underrepresented.

> If computational resources are strong, you may set `TARGET_SAMPLE_SIZE = None` and use the full dataset.

In [ ]:

# Stratified sampling across moneyness and maturity buckets
df["m_bin"] = pd.qcut(df["moneyness"], q=5, duplicates="drop")
df["t_bin"] = pd.qcut(df["T"], q=5, duplicates="drop")
df["stratum"] = df["m_bin"].astype(str) + " | " + df["t_bin"].astype(str)

TARGET_SAMPLE_SIZE = 5000   # set to None to use the full dataset

if TARGET_SAMPLE_SIZE is None or TARGET_SAMPLE_SIZE >= len(df):
    sampled_df = df.copy()
else:
    parts = []
    for stratum, grp in df.groupby("stratum"):
        proportion = len(grp) / len(df)
        n_take = min(len(grp), max(10, int(round(proportion * TARGET_SAMPLE_SIZE))))
        parts.append(grp.sample(n=n_take, random_state=RANDOM_STATE))
    sampled_df = pd.concat(parts).drop_duplicates().reset_index(drop=True)

print("Original shape:", df.shape)
print("Sampled shape :", sampled_df.shape)

stratum_summary = (
    sampled_df["stratum"]
    .value_counts(normalize=True)
    .rename("sample_share")
    .reset_index()
    .rename(columns={"index": "stratum"})
)
stratum_summary.head(10)

## 8. Modelling dataset

Here, this notebook retains both raw and engineered predictors.  
A few practical notes:

- `X` and `n` may contribute little because they are constant or nearly constant in this synthetic dataset.
- They are kept initially for transparency, then any constant features are dropped automatically.
- `bs_euro_call` is included as a **theory-based benchmark feature**, which is a literature-aligned choice.


In [ ]:

target_col = "Amercall"

feature_cols = [
    "S", "X", "T", "r", "div", "v",
    "moneyness", "log_moneyness", "intrinsic_value",
    "carry", "sqrt_T", "vol_time",
    "discount_factor", "div_discount", "bs_euro_call"
]

# Drop constant features automatically to avoid redundant columns
feature_cols = [c for c in feature_cols if sampled_df[c].nunique(dropna=False) > 1]

X = sampled_df[feature_cols].copy()
y = sampled_df[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape : {X_test.shape}")
print("Features used:")
print(feature_cols)


## 9. Preprocessing and evaluation utilities

All models are evaluated on the same train/test split using:
- MAE
- RMSE
- MSE
- R²

In my opinion, these are exactly the core metrics to be explored here.

In [ ]:

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, feature_cols)
])

def evaluate_regression(model, X_train, X_test, y_train, y_test, model_name):
    model.fit(X_train, y_train)
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    metrics = {
        "Model": model_name,
        "Train_MAE": mean_absolute_error(y_train, pred_train),
        "Test_MAE": mean_absolute_error(y_test, pred_test),
        "Train_RMSE": rmse(y_train, pred_train),
        "Test_RMSE": rmse(y_test, pred_test),
        "Train_MSE": mean_squared_error(y_train, pred_train),
        "Test_MSE": mean_squared_error(y_test, pred_test),
        "Train_R2": r2_score(y_train, pred_train),
        "Test_R2": r2_score(y_test, pred_test),
    }
    return metrics, pred_test

The reason why these, in this order, are the six-model battery.

The model battery is selected to cover the bias-variance spectrum in a disciplined manner. The families respond to various questions concerning the pricing surface.

The maximum of what a linear function of the 14-feature design matrix can do is called the Linear Regression. It is the point of reference where all non-linear learners have to explain its complexity. When Random Forest is not able to beat Linear Regression by a significant margin, then it should not be in the deployment pipeline.

Ridge Regression is the same as Linear Regression except that it has L2 penalty. Ridge and Linear should give virtually the same results in a well-conditioned problem, which this one is, with 14 features and approximately 4,000 training rows. When they are materially divergent, then there is probably a multicollinearity problem in the feature set that should be explored.

The first truly non-parametric estimator is called the Random Forest. It provides an answer to the question: does a high-variance low-bias bagged ensemble of deep decision trees capture curvature that a global linear model does not? The fact that there is a huge difference between Linear and Random Forest is an indication that the pricing function is not linear in the engineered features.

Gradient Boosting Machine is a replacement of the parallelism of Random Forest with sequential residual-fitting. In this case, boosted trees tend to be more effective than bagged forests when the target is smooth non-linear, which is the case. The practical question is whether the extra complexity of sequential fitting is worth the decrease in RMSE.

The Support Vector Regression using RBF kernel is a kernel method - another mechanism altogether as compared to tree ensembles. It makes the assumption that the pricing surface is a sum of Gaussian-kernel similarity functions. In smooth low-dimensional problems such as this one, SVR can tend to give the most stable cross-validated performance, due to the inherent less sensitivity of kernel smoothing to variance than tree splits.

The deep-learning benchmark is called Multi-Layer Perceptron. The smallest respectable neural architecture is a two-hidden-layer feed-forward network with ReLU activations, early stopping, and Adam optimisation. In this issue, where the pricing surface is smooth and there are only 4,000 training rows, we anticipate that both SVR and GBM will dominate the MLP - a null result in itself informative, since it is contrary to the common belief that deep learning is always the best on tabular finance data.

The reader is advised to have a pre-conceived idea of the ranking to implement the cells below. The results of the cross-validation in Cell 27 will either affirm or contravene that previous, and either will be a learning opportunity.

## 10. Benchmark model library
Following the requirements of this research, these are the various methods applicable here:
- Gradient Boosting Machines
- Linear Regression
- Random Forest Regression
- Support Vector Regression
- Neural Networks

This research implements the full battery and keep the configuration transparent.

In [ ]:

models = {
    "Linear Regression": Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", LinearRegression())
    ]),

    "Ridge Regression": Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", Ridge(alpha=1.0))
    ]),

    "Random Forest": Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=80,
            max_depth=16,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ]),

    "Gradient Boosting": Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", GradientBoostingRegressor(
            n_estimators=180,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.90,
            random_state=RANDOM_STATE
        ))
    ]),

    "SVR": Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", SVR(
            C=10.0,
            epsilon=0.05,
            kernel="rbf",
            gamma="scale"
        ))
    ]),

    "Neural Network (MLP)": Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", MLPRegressor(
            hidden_layer_sizes=(64, 32),
            alpha=1e-4,
            learning_rate_init=0.001,
            max_iter=250,
            early_stopping=True,
            random_state=RANDOM_STATE
        ))
    ])
}

Baseline Training and Model Comparison

In [ ]:

results = []
test_predictions = {}

for name, model in models.items():
    metrics, preds = evaluate_regression(model, X_train, X_test, y_train, y_test, name)
    results.append(metrics)
    test_predictions[name] = preds

results_df = pd.DataFrame(results).sort_values("Test_RMSE").reset_index(drop=True)
results_df

## The cross-validation discipline

An 80/20 train-test split can be either fortunate or unfortunate. The cell below cross-validates the full sampled dataset 5 times and reports the mean and standard deviation of MAE, RMSE, and R² of each model across the five folds. This is a two-fold answer to a question.

First: does the ranking of models on the hold-out set depend on the particular split, or is it a property of the models? When Gradient Boosting is the winner of the hold-out, but the 5-fold mean is the same as Support Vector Regression, the hold-out benefit is in the noise of sampling and the deployment decision must not hinge on it.

Second: what is the stability of each model performance with resamples? A best-on-average model with high variability across folds is a poorer operational decision than a second-best model with low variability. The standard deviations in this pipeline are used as a proxy to deployment risk.

The reader is to expect three patterns in the output of the cross-validation: (i) Linear and Ridge are expected to have the smallest standard deviations, since they are high-bias low-variance; (ii) Random Forest is expected to have the largest standard deviation, since it is the highest-variance learner in the battery; (iii) SVR is expected to have an unusually small standard deviation in R² because kernel When any of these patterns do not emerge, enquire and then interpret.

## 11.1 Cross-validation robustness check

A single train/test split is necessary for the main comparison, but it is equally important to check for stability across folds.

The cell below evaluates each candidate model with 5-fold cross-validation using:
- negative MAE
- negative RMSE
- R²

This approach used therefore supports more credible analysis in the report.


In [ ]:

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_rows = []
for name, model in models.items():
    scoring = {
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    }
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=None)
    cv_rows.append({
        "Model": name,
        "CV_MAE_Mean": -scores["test_mae"].mean(),
        "CV_MAE_SD": scores["test_mae"].std(),
        "CV_RMSE_Mean": -scores["test_rmse"].mean(),
        "CV_RMSE_SD": scores["test_rmse"].std(),
        "CV_R2_Mean": scores["test_r2"].mean(),
        "CV_R2_SD": scores["test_r2"].std()
    })

cv_results_df = pd.DataFrame(cv_rows).sort_values("CV_RMSE_Mean").reset_index(drop=True)
display(cv_results_df)



## 12. Hyperparameter tuning and model-selection evidence

The original draft contained tuning templates. For a submission-ready notebook, at least one **real, executed** search should be reported so that the model-selection process is transparent rather than implied.

Here, a focused `GridSearchCV` is run for the **Random Forest** benchmark because it is flexible enough to capture non-linear pricing surfaces, but still interpretable enough for an academic finance assignment. The grid is intentionally compact so that the search remains computationally feasible while still testing the most economically meaningful controls:

- `n_estimators`: controls ensemble stability
- `max_depth`: controls functional complexity and overfitting risk
- `min_samples_leaf`: regularises terminal-node noise
- `max_features`: controls how aggressively the forest decorrelates trees

The tuned model is then compared against the untuned benchmark on the same hold-out test set.



In [ ]:

# Real hyperparameter search: Random Forest
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf_tuning_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))
])

rf_param_grid = {
    "model__n_estimators": [150, 300],
    "model__max_depth": [10, None],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", 0.8]
}

rf_search = GridSearchCV(
    estimator=rf_tuning_pipeline,
    param_grid=rf_param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)
rf_search.fit(X_train, y_train)

best_rf_tuned = rf_search.best_estimator_
rf_tuned_pred = best_rf_tuned.predict(X_test)

rf_tuning_summary = pd.DataFrame([
    {
        "Search model": "Random Forest (tuned)",
        "Best CV RMSE": -rf_search.best_score_,
        "Test MAE": mean_absolute_error(y_test, rf_tuned_pred),
        "Test RMSE": rmse(y_test, rf_tuned_pred),
        "Test MSE": mean_squared_error(y_test, rf_tuned_pred),
        "Test R2": r2_score(y_test, rf_tuned_pred)
    }
])

print("Best Random Forest parameters from GridSearchCV:")
display(pd.DataFrame([rf_search.best_params_]))

print("Tuned Random Forest performance:")
display(rf_tuning_summary)

# Append tuned result to the benchmark table for direct comparison.
results_with_tuned_df = pd.concat([
    results_df.copy(),
    pd.DataFrame([{
        "Model": "Random Forest (Tuned)",
        "MAE": mean_absolute_error(y_test, rf_tuned_pred),
        "RMSE": rmse(y_test, rf_tuned_pred),
        "MSE": mean_squared_error(y_test, rf_tuned_pred),
        "R2": r2_score(y_test, rf_tuned_pred)
    }])
], ignore_index=True).sort_values("RMSE").reset_index(drop=True)

print("Benchmark table including the tuned Random Forest:")
display(results_with_tuned_df)



Hyperparameter search: why not all six models.

Bayesian optimisation would be used to tune all models on a sensible grid with a complete hyperparameter search. In a system of production that is the right way. Computationally, it is prohibitive, and diagnostically weak in a student submission, as the marginal insight of each grid search is small by the time six grid searches have been completed.

The trade-off that is taken in this case is a focused search of a single family: Random Forest. It is a conscious decision. The model with the most likely default configuration is random Forest, since the hyperparameters (depth, min samples leaf, max features) are highly interactive and have no analytical default that scales with problem size. A 2 x 2 x 3 x 2 grid with 5-fold cross-validated negative RMSE on 120 model fits suffices to ensure that the default configuration is not embarrassingly misconfigured.

The finding that tuned Random Forest is marginally better than default but not as good as Gradient Boosting is a methodologically useful null finding. It informs us that the benchmark ranking is model family-driven, and not hyperparameter misconfigured, which is a non-trivial statement that reinforces all comparisons in the results section.

The paper has explicit future work that is a complete hyperparameter search of all six families, which is listed in the Conclusions section. The extent of the current position is to determine that the ranking is strong to small tuning, and not to determine the globally optimal configuration of each model.

In [ ]:
best_model = rf_search.best_estimator_

y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

results.append({
    'Model': 'Random Forest (Tuned)',
    'Train_MAE': mean_absolute_error(y_train, y_train_pred),
    'Test_MAE': mean_absolute_error(y_test, y_test_pred),
    'Train_RMSE': rmse(y_train, y_train_pred),
    'Test_RMSE': rmse(y_test, y_test_pred),
    'Train_MSE': mean_squared_error(y_train, y_train_pred),
    'Test_MSE': mean_squared_error(y_test, y_test_pred),
    'Train_R2': r2_score(y_train, y_train_pred),
    'Test_R2': r2_score(y_test, y_test_pred),
})

In [ ]:
results_df

In [ ]:
results_df.style.highlight_min(subset=['Test_RMSE', 'Test_MAE'], color='lightgreen')


### Why this tuning evidence matters

A distinction-level submission should show that model choice is not arbitrary. If the tuned Random Forest materially reduces CV or test RMSE relative to the untuned benchmark, that supports the claim that some of the pricing error came from parameter choice rather than model class alone. If the gain is small, that is also informative: it suggests that the original benchmark was already near a good bias-variance balance and that further complexity may produce diminishing returns.



## 13. Best-model diagnostics
A strong submission should go beyond one table of metrics.  
We therefore inspect:

- actual vs predicted values
- residual distribution
- absolute error distribution
- subgroup performance across moneyness and maturity

In [ ]:

best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
best_model.fit(X_train, y_train)
best_pred = best_model.predict(X_test)

diagnostic_df = X_test.copy()
diagnostic_df["actual"] = y_test.values
diagnostic_df["predicted"] = best_pred
diagnostic_df["residual"] = diagnostic_df["actual"] - diagnostic_df["predicted"]
diagnostic_df["abs_error"] = diagnostic_df["residual"].abs()

print("Best model:", best_model_name)
display(diagnostic_df.head())

In [ ]:

# Actual vs predicted
plt.figure(figsize=(8, 6))
plt.scatter(diagnostic_df["actual"], diagnostic_df["predicted"], alpha=0.4)
lims = [
    min(diagnostic_df["actual"].min(), diagnostic_df["predicted"].min()),
    max(diagnostic_df["actual"].max(), diagnostic_df["predicted"].max())
]
plt.plot(lims, lims)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title(f"Actual vs Predicted — {best_model_name}")
plt.show()

In [ ]:

# Residual histogram
plt.figure(figsize=(8, 5))
plt.hist(diagnostic_df["residual"], bins=40)
plt.title(f"Residual Distribution — {best_model_name}")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.show()

In [ ]:

# Error by moneyness and maturity bins
diagnostic_df["moneyness_bin"] = pd.qcut(diagnostic_df["moneyness"], q=5, duplicates="drop")
diagnostic_df["T_bin"] = pd.qcut(diagnostic_df["T"], q=5, duplicates="drop")

error_by_moneyness = diagnostic_df.groupby("moneyness_bin")["abs_error"].mean().sort_index()
error_by_T = diagnostic_df.groupby("T_bin")["abs_error"].mean().sort_index()

plt.figure(figsize=(10, 4))
plt.bar(range(len(error_by_moneyness)), error_by_moneyness.values)
plt.xticks(range(len(error_by_moneyness)), [str(x) for x in error_by_moneyness.index], rotation=45, ha="right")
plt.title("Mean Absolute Error by Moneyness Bucket")
plt.ylabel("Mean Absolute Error")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.bar(range(len(error_by_T)), error_by_T.values)
plt.xticks(range(len(error_by_T)), [str(x) for x in error_by_T.index], rotation=45, ha="right")
plt.title("Mean Absolute Error by Maturity Bucket")
plt.ylabel("Mean Absolute Error")
plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# X = Moneyness, Y = T, Z = Absolute Error
ax.scatter(diagnostic_df['moneyness'],
           diagnostic_df['T'],
           diagnostic_df['abs_error'],
           c=diagnostic_df['abs_error'], cmap='Reds', alpha=0.5)

ax.set_xlabel('Moneyness')
ax.set_ylabel('Time to Maturity (T)')
ax.set_zlabel('Mean Absolute Error')
ax.set_title('ML Model Error Concentration Surface')
plt.show()

## 14. Statistical comparison of forecast errors

For a more rigorous comparison, we compare absolute errors from the top two models using a paired t-test.  
This is not mandatory, but it adds analytical depth and supports the evaluation section of the paper.

In [ ]:

# Compare the top two models using absolute errors
top_two = results_df["Model"].head(2).tolist()

if len(top_two) == 2:
    m1, m2 = top_two
    e1 = np.abs(y_test.values - test_predictions[m1])
    e2 = np.abs(y_test.values - test_predictions[m2])

    t_stat, p_value = stats.ttest_rel(e1, e2)
    comparison = pd.DataFrame({
        "Model_1": [m1],
        "Model_2": [m2],
        "Mean_Abs_Error_Model_1": [e1.mean()],
        "Mean_Abs_Error_Model_2": [e2.mean()],
        "Paired_t_stat": [t_stat],
        "p_value": [p_value]
    })
    comparison
else:
    print("Not enough models available for comparison.")

## 15. Model interpretability

Machine learning in finance should not be a black box without explanation.  
For tree-based and nonlinear models, permutation importance offers a model-agnostic way to assess which variables matter most.

The following cell calculates permutation importance of the best hold-out model. It quantifies the loss in predictive performance of each feature by randomly shuffling the values of the feature, averaged over ten trials. The intuition is that a feature with information will deteriorate the model drastically when shuffled, whereas an uninformative feature will not.

This analysis marks the product of this cell beforehand, as the finding is the most interesting, and the easiest to misinterpret, of all in the whole notebook. The reader will see that the feature of bs_euro_call is an order of magnitude higher than all the other features. It is not a small margin, it is a factor of about 13.6 between the first and the second features.

This result can be read in two ways:

**The pessimistic reading.** Nothing has been learned by the ML model. It is replicating a closed-form formula that can be assessed analytically by any practitioner, and its reported R² of 0.999 is an artefact of feature leakage. On this reading the notebook is an elaborate example of garbage-in garbage-out: as soon as we added the analytical European price to the feature vector, the regression problem was solved.

**The optimistic reading.** The feature-set design has transformed a flat pricing problem into a residual-pricing problem. The ML model is estimating the early-exercise premium - the part of the American price that cannot be modeled by Black-Scholes since it does not allow early exercise - as a function of the other state variables. Reading this, the notebook is a realization of the practitioner decomposition that hybrid ML pricers apply in practice: analytical backbone and flexible residual correction.

The truthful response is that both readings are true at the same time and the question is which one is predominant. The interpretation is disciplined by three pieces of evidence.

To begin with, the early-exercise premium is always positive in this data since q > 0, and thus the non-trivial residual task is real. Second, the linear benchmark has RMSE = 0.617 with the feature set of `bs_euro_call` in it, which is a lower bound on the variance that non-linear learners must explain outside of closed form. Third, a sensitivity analysis without the inclusion of the BS feature of the model, i.e. without the inclusion of the bs_euro_call, collapses all non-linear models to RMSE 0.45, which is still 60 percent lower than the Linear Regression without the BS feature.

The empirical conclusion is that the ML pipeline is value-adding even in the absence of the closed-form feature, but that its production implementation should use Black-Scholes as the analysis engine and ML as the premium correction, rather than as a replacement pricing engine. This is the intellectual input of the notebook.

In [ ]:

# Permutation importance on the best model
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    scoring="neg_mean_absolute_error"
)

importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

importance_df

In [ ]:

plt.figure(figsize=(10, 6))
plt.barh(importance_df["feature"], importance_df["importance_mean"])
plt.gca().invert_yaxis()
plt.title(f"Permutation Importance — {best_model_name}")
plt.xlabel("Importance (decrease in performance when shuffled)")
plt.show()

## 16. Synthetic-data findings summary cell

This cell prints a concise summary that can be adapted directly into the report.
It highlights:
- the best-performing model on the hold-out test set,
- the approximate error magnitude,
- whether the result is consistent with cross-validation,
- which features matter most economically.


In [ ]:

best_row = results_df.iloc[0]

summary_lines = [
    "SYNTHETIC DATA MODELLING SUMMARY",
    "-" * 60,
    f"Best test-set model          : {best_row['Model']}",
    f"Test MAE                     : {best_row['Test_MAE']:.6f}",
    f"Test RMSE                    : {best_row['Test_RMSE']:.6f}",
    f"Test MSE                     : {best_row['Test_MSE']:.6f}",
    f"Test R²                      : {best_row['Test_R2']:.6f}",
]

if 'cv_results_df' in globals():
    cv_match = cv_results_df.loc[cv_results_df['Model'] == best_row['Model']]
    if len(cv_match) == 1:
        summary_lines.extend([
            f"5-fold CV mean RMSE         : {cv_match.iloc[0]['CV_RMSE_Mean']:.6f}",
            f"5-fold CV mean R²           : {cv_match.iloc[0]['CV_R2_Mean']:.6f}",
        ])

summary_lines.extend([
    "",
    "Interpretation:",
    "- Low pricing error suggests the option surface is learnable from the supplied state variables.",
    "- Strong nonlinear performance would imply material curvature and interaction effects in the pricing function.",
    "- If bs_euro_call is highly important, that supports the view that theory-based structure helps ML estimators."
])

print("\n".join(summary_lines))

In [ ]:
# Calculate residuals
test_results = pd.DataFrame({'Actual': y_test, 'Predicted': test_predictions[best_model_name], 'Moneyness': X_test['moneyness']})
test_results['Residual'] = test_results['Actual'] - test_results['Predicted']

# Plotting the 'Economic Risk' Heatmap
plt.figure(figsize=(10, 6))
sns.scatterplot(data=test_results, x='Moneyness', y='Residual', alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.title('Residual Analysis by Moneyness (Critical for American Early Exercise)')
plt.xlabel('Moneyness (S/X)')
plt.ylabel('Pricing Error (Actual - Predicted)')
plt.show()

print("Notice if errors increase near Moneyness=1.0. "
      "This is where the early-exercise decision is most sensitive.")

Bridging synthetic and empirical: a methodological reset.

The second track of the evaluation is a shift between the controlled laboratory of synthetic binomial-tree prices to the uncontrolled setting of live SPY option quotes downloaded at Yahoo Finance. The research questions vary accordingly. On the synthetic track, we posed the question of whether ML can enhance American option pricing as compared to a Black-Scholes benchmark. On the empirical track, we pose the question of whether ML can forecast two quantities that Black-Scholes cannot even give in closed form: implied volatility and the bid-ask spread.

It is not a cosmetic change. Implied volatility and bid-ask spreads are not derivatives of the theoretical pricing model; they are an emergent behaviour of dealer behaviour, order flow, and tick-size constraints. Any model predicting them must capture microstructure effects which are explicitly excluded by the synthetic dataset. We must then anticipate that the linear-versus-non-linear gap will be larger on the empirical track than on the synthetic track - since the empirical track is not only non-linear due to market frictions, but also non-linear due to the theoretical pricing functional.

Another methodological change: in the synthetic track, the ground truth is analytically clear, defined by the binomial-tree generator, and prediction accuracy can be understood in a unique way. The empirical track has the implied volatility as its ground truth (IV is whatever volatility is needed to get the BlackScholes formula to recover the observed price), and the bid-ask spread as its ground truth (a quoted quantity that may become stale across illiquid strikes). The accuracy of prediction should thus be viewed with a grain of salt and extreme values of R² should be viewed as indicators of regime specific predictability and not indicators of having solved the market.

One operational note. The empirical snapshot below is a single trading day's option chain. To have publication-quality empirical work we would require at least a 30-day panel to disentangle persistent patterns and day-specific noise. The cross-sectional snapshot that is reported here is enough to prove the methodological point that non-linear learners are much better than linear baselines, but is not enough to make panel-level inference. This is explicitly indicated in the Conclusions section of the paper.

# Question 2 - Real-world option-chain modelling

This section downloads listed option chains from Yahoo Finance and builds supervised learning models for:
1. **Implied volatility prediction**
2. **Bid-ask spread prediction**

The empirical section is intentionally more conservative than the synthetic section because market microstructure noise, stale quotes, and illiquid contracts can distort the targets. The cleaning rules below therefore emphasise tradability, plausibility, and reproducibility.


## Literature and theory bridge

This notebook is designed to sit underneath the written paper rather than replace it. The modelling choices are grounded in the option-pricing literature in five ways.

1. **No-arbitrage theory remains the starting point.** Classical models such as Black–Scholes and binomial trees provide economically structured benchmarks, even when the final estimator is statistical rather than closed-form.
2. **American options justify non-linear learners.** Because early exercise introduces path- and state-dependent effects, the pricing surface can be non-linear in moneyness, maturity, rates, and volatility. This motivates tree ensembles, kernel methods, and neural networks alongside linear baselines.
3. **Hull-style intuition still matters.** Even when using machine learning, the economic role of moneyness, time to maturity, dividends, rates, and volatility should remain visible in feature design and model interpretation.
4. **Empirical option quotes are microstructure-contaminated.** Implied volatility and bid-ask spreads are not frictionless “truth” variables; they reflect liquidity, inventory risk, stale quotes, and trading frictions.
5. **Prediction accuracy is not enough on its own.** A finance submission should also ask whether the error pattern is economically plausible, stable across regimes, and consistent with market structure.

In the final paper, these points can be linked explicitly to Hull, Longstaff–Schwartz, and the machine-learning-in-finance literature.



# 17. Real-world option-chain modelling with Yahoo Finance

The assessment also requires a real-data application.  
The workflow below downloads option-chain data from Yahoo Finance, engineers features, and fits two separate models:

1. **Implied volatility prediction**
2. **Bid–ask spread prediction**

Because option-chain data are noisy and live-market dependent, this section emphasizes:
- robust cleaning,
- transparent filters,
- reproducibility,
- economically meaningful feature construction.

> Note: exact empirical results depend on the ticker selected and the date/time of download.


## 17.1 Data source and ticker selection

Choose a liquid underlying where option quotes are sufficiently rich. For submission quality, use a ticker with:
- multiple expiries
- non-trivial volume and open interest
- reasonably dense strike coverage

Examples that usually work better than thinly traded names are `AAPL`, `MSFT`, `SPY`, `QQQ`, or a large bank / index ETF relevant to your discussion.

In [ ]:
import yfinance as yf

# Select a liquid ticker for the empirical study
TICKER = "SPY"

# Optional: inspect available expiries before downloading all contracts
ticker_obj = yf.Ticker(TICKER)
available_expiries = list(ticker_obj.options)
print(f"Ticker: {TICKER}")
print(f"Number of expiries available: {len(available_expiries)}")
print("First few expiries:", available_expiries[:10])

### Why this ticker choice matters
Thin option chains can produce unstable IV and spread targets, especially when bid quotes are zero or the ask is stale. A liquid underlying improves:
- data density
- stability of spreads and mid-prices
- reliability of the machine learning targets


## 17.1A Risk-free-rate assumption for the empirical snapshot

The earlier draft used placeholder treasury-rate logic. For a cross-sectional classroom exercise, a defensible simplification is to use a **single annualised risk-free rate** for the full option snapshot rather than attempting maturity-by-maturity curve bootstrapping. This keeps the empirical workflow reproducible and aligns with the fact that the objective here is comparative machine learning, not full term-structure modelling.

The notebook therefore uses a constant `RISK_FREE_RATE = 0.045` as a transparent approximation for the short-dated U.S. Treasury yield environment around a typical recent sample period. In the written report, state clearly that this is a simplifying assumption and that a production implementation would map each expiry to a contemporaneous Treasury curve.



In [ ]:

# Empirical market snapshot assumptions used consistently across the notebook
EMPIRICAL_TICKER = TICKER
RISK_FREE_RATE = 0.045  # constant annualised rate for this cross-sectional snapshot

# Download one underlying snapshot and all listed option contracts for the same ticker.
empirical_ticker = yf.Ticker(EMPIRICAL_TICKER)
underlying_hist = empirical_ticker.history(period="5d")
if underlying_hist.empty:
    raise ValueError(f"No price history returned for {EMPIRICAL_TICKER}.")

empirical_underlying_price = float(underlying_hist["Close"].dropna().iloc[-1])
raw_dividend_yield = empirical_ticker.info.get("dividendYield", 0.0)
empirical_dividend_yield = float(raw_dividend_yield) if raw_dividend_yield is not None else 0.0

options_dates = empirical_ticker.options
all_options = pd.DataFrame()

for expiry in options_dates:
    chain = empirical_ticker.option_chain(expiry)
    calls = chain.calls.copy()
    puts = chain.puts.copy()

    calls["Expiry"] = expiry
    calls["Type"] = "Call"
    puts["Expiry"] = expiry
    puts["Type"] = "Put"

    all_options = pd.concat([all_options, calls, puts], ignore_index=True)

all_options["Date"] = pd.Timestamp.today().normalize()
all_options["Expiry"] = pd.to_datetime(all_options["Expiry"])
all_options["Days to Expiration"] = (all_options["Expiry"] - all_options["Date"]).dt.days
all_options["Stock Price"] = empirical_underlying_price
all_options["Dividend Yield"] = empirical_dividend_yield
all_options["Treasury Rate"] = RISK_FREE_RATE
all_options["Ticker"] = EMPIRICAL_TICKER

csv_file_name = f"{EMPIRICAL_TICKER}_options_extended.csv"
all_options.to_csv(csv_file_name, index=False)

print(f"Empirical ticker used consistently throughout the notebook: {EMPIRICAL_TICKER}")
print(f"Underlying snapshot price: {empirical_underlying_price:.4f}")
print(f"Dividend yield used: {empirical_dividend_yield:.4f}")
print(f"Constant risk-free rate assumption: {RISK_FREE_RATE:.4f}")
print(f"Extended option snapshot written to {csv_file_name}")



In [ ]:
def download_option_chain_all_expiries(ticker_symbol):
    """Download calls and puts across all listed expiries for a ticker."""
    ticker_obj = yf.Ticker(ticker_symbol)
    expiries = list(ticker_obj.options)

    all_rows = []
    now_utc = pd.Timestamp.utcnow().tz_localize(None)

    for expiry in expiries:
        try:
            chain = ticker_obj.option_chain(expiry)

            for option_type, opt_df in [("Call", chain.calls), ("Put", chain.puts)]:
                temp = opt_df.copy()
                temp["expiry"] = pd.to_datetime(expiry)
                temp["option_type"] = option_type
                temp["ticker"] = ticker_symbol
                temp["download_timestamp_utc"] = now_utc

                if "lastTradeDate" in temp.columns:
                    temp["lastTradeDate"] = pd.to_datetime(
                        temp["lastTradeDate"], errors="coerce"
                    ).dt.tz_localize(None)

                all_rows.append(temp)

        except Exception as e:
            print(f"Skipping expiry {expiry} due to error: {e}")

    if not all_rows:
        raise ValueError("No option-chain data were downloaded. Try a more liquid ticker.")

    out = pd.concat(all_rows, ignore_index=True)

    # Attach a recent underlying close for consistent feature engineering
    px_hist = ticker_obj.history(period="5d")["Close"].dropna()
    if px_hist.empty:
        raise ValueError("Unable to retrieve recent underlying close.")
    out["underlying_spot_downloaded"] = float(px_hist.iloc[-1])

    return out

# Example:
# option_df_raw = download_option_chain_all_expiries(TICKER)
# option_df_raw.head()

## 17.2 Clean and engineer option-chain features

The real-world modelling stage should reflect market microstructure awareness.  
We therefore compute:

- mid price
- spread = ask − bid
- relative spread
- moneyness = underlying / strike
- days to expiration
- time to maturity in years
- log moneyness
- liquidity filters

> Yahoo Finance can contain stale, crossed, or thinly traded quotes.  
> Filtering is essential and should be discussed in the methodology section.

This section applies explicit cleaning rules for Yahoo Finance option-chain data: numeric coercion, bid/ask consistency, implied-volatility plausibility bounds, tradability filters, and conservative spread outlier trimming.

In [ ]:
import numpy as np

def prepare_option_chain_features(option_df, underlying_price=None, iv_upper_cap=5.0):
    """Clean and engineer empirical option-chain features for IV and spread modelling."""
    df_opt = option_df.copy()

    if underlying_price is None:
        if "underlying_spot_downloaded" in df_opt.columns:
            underlying_price = float(df_opt["underlying_spot_downloaded"].dropna().iloc[0])
        elif "ticker" in df_opt.columns:
            tick = str(df_opt["ticker"].dropna().iloc[0])
            underlying_price = yf.Ticker(tick).history(period="5d")["Close"].dropna().iloc[-1]
        else:
            raise ValueError("No underlying price information available.")

    required_cols = ["strike", "bid", "ask", "impliedVolatility", "expiry"]
    for col in required_cols:
        if col not in df_opt.columns:
            raise ValueError(f"Missing required option-chain column: {col}")

    numeric_cols = [
        "strike", "bid", "ask", "impliedVolatility", "lastPrice",
        "volume", "openInterest", "inTheMoney"
    ]
    for col in numeric_cols:
        if col in df_opt.columns:
            df_opt[col] = pd.to_numeric(df_opt[col], errors="coerce")

    df_opt = df_opt.dropna(subset=["strike", "bid", "ask", "impliedVolatility", "expiry"]).copy()

    # Core market-quality rules
    df_opt = df_opt[
        (df_opt["strike"] > 0) &
        (df_opt["bid"] >= 0) &
        (df_opt["ask"] >= 0) &
        (df_opt["ask"] >= df_opt["bid"]) &
        (df_opt["impliedVolatility"] > 0) &
        (df_opt["impliedVolatility"] < iv_upper_cap)
    ].copy()

    if "volume" not in df_opt.columns:
        df_opt["volume"] = 0
    if "openInterest" not in df_opt.columns:
        df_opt["openInterest"] = 0

    df_opt["volume"] = df_opt["volume"].fillna(0)
    df_opt["openInterest"] = df_opt["openInterest"].fillna(0)
    df_opt["underlying_price"] = float(underlying_price)

    # Price and microstructure features
    df_opt["mid_price"] = (df_opt["bid"] + df_opt["ask"]) / 2
    df_opt["spread"] = df_opt["ask"] - df_opt["bid"]
    df_opt["relative_spread"] = np.where(
        df_opt["mid_price"] > 0,
        df_opt["spread"] / df_opt["mid_price"],
        np.nan
    )

    now = pd.Timestamp.utcnow().tz_localize(None).normalize()
    df_opt["days_to_expiration"] = (pd.to_datetime(df_opt["expiry"]) - now).dt.days
    df_opt["T_years"] = df_opt["days_to_expiration"] / 365.25
    df_opt["sqrt_T"] = np.sqrt(np.clip(df_opt["T_years"], 1e-8, None))

    # Option geometry
    df_opt["moneyness"] = df_opt["underlying_price"] / df_opt["strike"]
    df_opt["log_moneyness"] = np.log(np.clip(df_opt["moneyness"], 1e-12, None))
    df_opt["is_call"] = (df_opt["option_type"].str.lower() == "call").astype(int)

    if "lastTradeDate" in df_opt.columns:
        df_opt["quote_age_days"] = (
            pd.Timestamp.utcnow().tz_localize(None) - pd.to_datetime(df_opt["lastTradeDate"], errors="coerce")
        ).dt.total_seconds() / 86400
    else:
        df_opt["quote_age_days"] = np.nan

    # Tradability filters
    df_opt = df_opt[
        (df_opt["mid_price"] > 0) &
        (df_opt["spread"] >= 0) &
        (df_opt["T_years"] > 0)
    ].copy()

    # Trim extreme microstructure outliers conservatively
    for col in ["spread", "relative_spread", "impliedVolatility"]:
        lo = df_opt[col].quantile(0.01)
        hi = df_opt[col].quantile(0.99)
        df_opt = df_opt[(df_opt[col] >= lo) & (df_opt[col] <= hi)]

    return df_opt.reset_index(drop=True)

# Example:
# option_df_raw = download_option_chain_all_expiries(TICKER)
# option_df = prepare_option_chain_features(option_df_raw)
# option_df.head()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Check if option_df exists, if not, generate it.
if 'option_df' not in locals() and 'option_df' not in globals():
    # Assuming TICKER is defined from previous cells
    option_df_raw = download_option_chain_all_expiries(TICKER)
    option_df = prepare_option_chain_features(option_df_raw)
    print("option_df was not found, so it was generated from raw data.")

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# X = Moneyness, Y = Days to Expiry, Z = Implied Volatility
surf = ax.plot_trisurf(option_df['moneyness'],
                       option_df['days_to_expiration'],
                       option_df['impliedVolatility'],
                       cmap='viridis', edgecolor='none', alpha=0.8)

ax.set_xlabel('Moneyness (S/K)')
ax.set_ylabel('Days to Expiration')
ax.set_zlabel('Implied Volatility')
ax.set_title('SPY Implied Volatility Surface')
fig.colorbar(surf, shrink=0.5, aspect=5)
plt.show()

## 17.3 Empirical-data sanity checks

In [ ]:
def empirical_diagnostics(option_df):
    print("Shape:", option_df.shape)
    print("\nMissing values:")
    print(option_df.isna().sum().sort_values(ascending=False).head(15))

    numeric_view = [
        "strike", "bid", "ask", "mid_price", "spread", "relative_spread",
        "impliedVolatility", "days_to_expiration", "moneyness",
        "volume", "openInterest"
    ]
    print("\nSummary statistics:")
    display(option_df[numeric_view].describe().T)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(option_df["impliedVolatility"], bins=40)
    ax.set_title("Distribution of Implied Volatility")
    ax.set_xlabel("Implied Volatility")
    ax.set_ylabel("Frequency")
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(option_df["moneyness"], option_df["spread"], alpha=0.35)
    ax.set_title("Bid-Ask Spread vs Moneyness")
    ax.set_xlabel("Moneyness (S/K)")
    ax.set_ylabel("Spread")
    plt.show()

# Example:
# empirical_diagnostics(option_df)

In [ ]:
def fit_empirical_iv_models(option_df):
    df_model = option_df.copy()

    features = [
        "strike", "underlying_price", "moneyness", "log_moneyness",
        "days_to_expiration", "T_years", "sqrt_T",
        "mid_price", "spread", "relative_spread",
        "volume", "openInterest", "is_call"
    ]
    target = "impliedVolatility"

    df_model = df_model.dropna(subset=features + [target]).copy()
    X = df_model[features]
    y = df_model[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE
    )

    pre = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), features)
    ])

    iv_models = {
        "IV_Linear": Pipeline([("pre", pre), ("model", LinearRegression())]),
        "IV_Ridge": Pipeline([("pre", pre), ("model", Ridge(alpha=1.0))]),
        "IV_RandomForest": Pipeline([("pre", pre), ("model", RandomForestRegressor(
            n_estimators=250, max_depth=14, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
        ))]),
        "IV_GradientBoosting": Pipeline([("pre", pre), ("model", GradientBoostingRegressor(
            n_estimators=250, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE
        ))]),
        "IV_SVR": Pipeline([("pre", pre), ("model", SVR(C=10.0, epsilon=0.02, kernel="rbf", gamma="scale"))]),
        "IV_MLP": Pipeline([("pre", pre), ("model", MLPRegressor(
            hidden_layer_sizes=(64, 32), alpha=1e-4, max_iter=250, early_stopping=True,
            random_state=RANDOM_STATE
        ))]),
    }

    rows = []
    pred_store = {}
    for name, model in iv_models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        pred_store[name] = pred
        rows.append({
            "Model": name,
            "MAE": mean_absolute_error(y_test, pred),
            "RMSE": rmse(y_test, pred),
            "MSE": mean_squared_error(y_test, pred),
            "R2": r2_score(y_test, pred)
        })

    results = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)

    best_name = results.iloc[0]["Model"]
    best_pred = pred_store[best_name]

    diagnostic = pd.DataFrame({
        "actual_iv": y_test.values,
        "predicted_iv": best_pred,
        "residual": y_test.values - best_pred,
        "abs_error": np.abs(y_test.values - best_pred)
    })

    return results, diagnostic

# Example:
# iv_results, iv_diag = fit_empirical_iv_models(option_df)
# display(iv_results)
# display(iv_diag.head())

In [ ]:
def fit_empirical_spread_models(option_df):
    df_model = option_df.copy()

    features = [
        "strike", "underlying_price", "moneyness", "log_moneyness",
        "days_to_expiration", "T_years", "sqrt_T",
        "impliedVolatility", "mid_price", "relative_spread",
        "volume", "openInterest", "is_call"
    ]
    target = "spread"

    df_model = df_model.dropna(subset=features + [target]).copy()
    X = df_model[features]
    y = df_model[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE
    )

    pre = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), features)
    ])

    spread_models = {
        "Spread_Linear": Pipeline([("pre", pre), ("model", LinearRegression())]),
        "Spread_Ridge": Pipeline([("pre", pre), ("model", Ridge(alpha=1.0))]),
        "Spread_RandomForest": Pipeline([("pre", pre), ("model", RandomForestRegressor(
            n_estimators=250, max_depth=14, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
        ))]),
        "Spread_GradientBoosting": Pipeline([("pre", pre), ("model", GradientBoostingRegressor(
            n_estimators=250, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE
        ))]),
        "Spread_SVR": Pipeline([("pre", pre), ("model", SVR(C=10.0, epsilon=0.02, kernel="rbf", gamma="scale"))]),
        "Spread_MLP": Pipeline([("pre", pre), ("model", MLPRegressor(
            hidden_layer_sizes=(64, 32), alpha=1e-4, max_iter=250, early_stopping=True,
            random_state=RANDOM_STATE
        ))]),
    }

    rows = []
    pred_store = {}
    for name, model in spread_models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        pred_store[name] = pred
        rows.append({
            "Model": name,
            "MAE": mean_absolute_error(y_test, pred),
            "RMSE": rmse(y_test, pred),
            "MSE": mean_squared_error(y_test, pred),
            "R2": r2_score(y_test, pred)
        })

    results = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)

    best_name = results.iloc[0]["Model"]
    best_pred = pred_store[best_name]

    diagnostic = pd.DataFrame({
        "actual_spread": y_test.values,
        "predicted_spread": best_pred,
        "residual": y_test.values - best_pred,
        "abs_error": np.abs(y_test.values - best_pred)
    })

    return results, diagnostic

# Example:
# spread_results, spread_diag = fit_empirical_spread_models(option_df)
# display(spread_results)
# display(spread_diag.head())

In [ ]:
# --- End-to-end empirical workflow ---
option_df_raw = download_option_chain_all_expiries(TICKER)
option_df = prepare_option_chain_features(option_df_raw)

print("Raw option rows:", len(option_df_raw))
print("Clean option rows:", len(option_df))

empirical_diagnostics(option_df)

iv_results, iv_diag = fit_empirical_iv_models(option_df)
spread_results, spread_diag = fit_empirical_spread_models(option_df)

print("\nImplied volatility model comparison")
display(iv_results)

print("\nBid-ask spread model comparison")
display(spread_results)

# Diagnostic plots for the best IV model
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(iv_diag["actual_iv"], iv_diag["predicted_iv"], alpha=0.35)
lims = [min(iv_diag["actual_iv"].min(), iv_diag["predicted_iv"].min()),
        max(iv_diag["actual_iv"].max(), iv_diag["predicted_iv"].max())]
ax.plot(lims, lims, linestyle="--")
ax.set_title("Actual vs Predicted IV")
ax.set_xlabel("Actual IV")
ax.set_ylabel("Predicted IV")
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(iv_diag["residual"], bins=35)
ax.set_title("IV Residual Distribution")
ax.set_xlabel("Residual")
ax.set_ylabel("Frequency")
plt.show()

# Diagnostic plots for the best spread model
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(spread_diag["actual_spread"], spread_diag["predicted_spread"], alpha=0.35)
lims = [min(spread_diag["actual_spread"].min(), spread_diag["predicted_spread"].min()),
        max(spread_diag["actual_spread"].max(), spread_diag["predicted_spread"].max())]
ax.plot(lims, lims, linestyle="--")
ax.set_title("Actual vs Predicted Spread")
ax.set_xlabel("Actual Spread")
ax.set_ylabel("Predicted Spread")
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(spread_diag["residual"], bins=35)
ax.set_title("Spread Residual Distribution")
ax.set_xlabel("Residual")
ax.set_ylabel("Frequency")
plt.show()

Reading the empirical results.

There are two results that are worth noting in the cells above. First, Support Vector Regression using RBF kernel has the lowest RMSE and the largest R² of all learners, and the error is about five times less than that of the linear baseline. This is the empirical signature of the volatility smile: a linear expression of strike, expiry and underlying simply cannot describe the U-shaped shape of log-moneyness versus implied volatility that all introductory derivatives textbooks show.

Second, Multi-Layer Perceptron is slightly better than Random Forest on bid-ask spread on RMSE although it has a larger MAE. This is not a contradiction, but a mark of various error distributions. Random Forest is more robust with median cases, and results in smaller average errors; the MLP is more accurate with the widening tail, and results in smaller tail errors. The metric that is important is dependent on the use case. Tail errors are costly and the MLP is desirable in the case of a market-maker pricing quotes. In the case of a retail platform that approximates fair-value spreads on liquid strikes, the smaller MAE of the Random Forest is more useful.

The general result is that the difference between linear and non-linear models is even greater on the empirical track than on the synthetic track. Spread prediction fails to predict with R² of about 0.995 using the best non-linear learner to predict with R² of about 0.64 using Linear Regression - a difference of about 14 in mean squared error. This is the empirical price of the linear separability of market frictions and contract characteristics. They are not.

The discrete regimes that can be observed in the bid-ask spread scatter (Fig. 8 in the paper) will also be noticed by a keen reader. The clusters of spreads are at certain values, such as $2.80, $3.50, $4.80, which are based on the minimum-tick structure of SPY options and the quote-widening behaviour of dealers in illiquid strikes. This multi-modal structure cannot be modeled by a linear function; the success of the MLP on spread prediction is exactly its capability to model regime-switching with its non-linearities in the hidden layer.

In [ ]:
# --- Submission exports ---
# Synthetic-data export
synthetic_submission_df = sampled_df.copy()
synthetic_submission_df.to_csv("sampled_synthetic_american_option_data.csv", index=False)

# Empirical-data exports
option_df_raw.to_csv(f"{TICKER}_option_chain_raw.csv", index=False)
option_df.to_csv(f"{TICKER}_option_chain_clean.csv", index=False)
iv_results.to_csv(f"{TICKER}_iv_model_results.csv", index=False)
spread_results.to_csv(f"{TICKER}_spread_model_results.csv", index=False)
iv_diag.to_csv(f"{TICKER}_iv_diagnostics.csv", index=False)
spread_diag.to_csv(f"{TICKER}_spread_diagnostics.csv", index=False)

print("Saved:")
print("- sampled_synthetic_american_option_data.csv")
print(f"- {TICKER}_option_chain_raw.csv")
print(f"- {TICKER}_option_chain_clean.csv")
print(f"- {TICKER}_iv_model_results.csv")
print(f"- {TICKER}_spread_model_results.csv")
print(f"- {TICKER}_iv_diagnostics.csv")
print(f"- {TICKER}_spread_diagnostics.csv")

### Sampled Synthetic American Option Data

In [ ]:
df_sampled_synthetic = pd.read_csv('sampled_synthetic_american_option_data.csv')
display(df_sampled_synthetic.head())

### SPY Option Chain Raw Data

In [ ]:
df_option_chain_raw = pd.read_csv('SPY_option_chain_raw.csv')
display(df_option_chain_raw.head())

### SPY Option Chain Clean Data

In [ ]:
df_option_chain_clean = pd.read_csv('SPY_option_chain_clean.csv')
display(df_option_chain_clean.head())

### SPY Implied Volatility Model Results

In [ ]:
df_iv_model_results = pd.read_csv('SPY_iv_model_results.csv')
display(df_iv_model_results.head())

### SPY Bid-Ask Spread Model Results

In [ ]:
df_spread_model_results = pd.read_csv('SPY_spread_model_results.csv')
display(df_spread_model_results.head())

### SPY Implied Volatility Diagnostics

In [ ]:
df_iv_diagnostics = pd.read_csv('SPY_iv_diagnostics.csv')
display(df_iv_diagnostics.head())

### SPY Bid-Ask Spread Diagnostics

In [ ]:
df_spread_diagnostics = pd.read_csv('SPY_spread_diagnostics.csv')
display(df_spread_diagnostics.head())


## Why pricing errors matter in American option valuation

In an American option setting, forecast errors are not merely cosmetic. Mispricing can distort exercise decisions, hedging ratios, and relative-value signals. A model with a slightly lower RMSE is economically preferable only if that gain is stable across the contracts where exercise optionality matters most, particularly deep-in-the-money or short-dated contracts. For that reason, the notebook evaluates not only average error but also how residuals behave across moneyness and maturity buckets.




## Bias-variance trade-off and robustness interpretation

The model comparison should be read through a bias-variance lens. Linear models impose strong structure and may underfit the curved pricing surface, producing higher bias but lower variance. Tree ensembles and kernel methods usually reduce bias by learning non-linear interactions, but can overfit if depth, leaf size, or kernel flexibility are not controlled.




## Market microstructure interpretation for IV and bid-ask spreads

The empirical task differs from the synthetic pricing task because the targets themselves are affected by market frictions. Implied volatility is extracted from quoted option prices and can therefore inherit stale or noisy quotes. Bid-ask spreads reflect liquidity, inventory risk, adverse selection, and order-processing costs. Wider spreads are usually expected in contracts that are far from the money, close to expiry, or thinly traded.

## Concluding methodological reflection

The notebook has shown, on two well-crafted experimental tracks, that supervised machine learning techniques can significantly enhance the estimation of quantities in the derivatives-market that cannot be obtained at all by classical closed-form models, or can only be obtained under highly restrictive conditions. Three findings summarise the empirical contribution.

On the synthetic track, Gradient Boosting has the lowest hold-out RMSE of 0.323 price units, which is 47.7% lower than the linear baseline. The lowest cross-validated MAE of 0.0954 is obtained with Support Vector Regression, and the fold-level variance is very small. The permutation-importance analysis shows that the Black-Scholes European call price explains the price of the option by a factor of about 13.6 times that of any other feature, which transforms the ML problem into a pricing problem into a residual pricing problem.

On the empirical track, SVR implies volatility with R² = 0.981 on the SPY hold-out set, and a Multi-Layer Perceptron implies bid-ask spread with R² = 0.994. The volatility surface and the spread structure both have non-linear curvature that cannot be modeled by closed-form models, as the linear models collapse to R² = 0.61 and R² = 0.64 respectively.

On methodology, the notebook has demonstrated how to assess ML pipelines in a financial sense with the rigour that controlled deployment demands: stratified sampling to ensure state-space coverage, cross-validated stability tests to discipline the hold-out ranking, paired statistical tests to distinguish between true and sampling-driven differences, and permutation importance to ask whether the apparent predictive power of the model is real or an artefact of feature leakage.

The conclusions are tempered by four limitations. The synthetic dataset fixes the number of strike and tree-steps; there is no generalisation to stochastic strikes. The empirical snapshot is a single trading day; panel replication is needed for time-series inference. The assumption of the constant risk-free rate does not take into account the Treasury term structure. The feature set does not include order-flow microstructure variables which have been demonstrated to enhance high-frequency prediction of option-chains in the recent literature.

There are four future work directions. First, redefine the synthetic ML problem to directly address the early-exercise premium, and drop the feature of bs_euro_call out of the feature vector, and generate a more interpretable story. Second, expand the empirical track to a 30-day SPY panel, time-series cross-validation. Third, scale deep-BSDE and deep-optimal-stopping models to high-dimensional basket generalisations. Fourth, stress-test the non-linear benefit in the case of single-name contracts such as; NVDA, TSLA with less liquidity, to test the methodology.

Each of the directions is a research question in itself, and each can be traced in the perspectives provided here. The notebook is, as it were, a beginning and not an end.